In [ ]:
# 1. Install a version of NumPy that works on Colab (Python 3.12)
# We force 1.26.4 because it is the bridge between the old code and new Python.
!pip install numpy==1.26.4

# 2. Install all the necessary libraries manually *before* chatterbox
# We do this to ensure we get the right versions without triggering the error.
!pip install "librosa>=0.11.0" "s3tokenizer" "torch>=2.6.0" "torchaudio>=2.6.0" "transformers>=4.46.3" "diffusers>=0.29.0" "resemble-perth>=1.0.1" "conformer>=0.3.2" "safetensors>=0.5.3" "spacy-pkuseg" "pykakasi>=2.3.0" "pyloudnorm" "omegaconf" "gradio"

# 3. Install chatterbox-tts WITHOUT dependencies
# This prevents it from trying to downgrade NumPy to the broken version.
!pip install chatterbox-tts --no-deps

print("✅ Installation complete. Please proceed to the next cell.")

In [ ]:
import os
import random
import numpy as np
import torch
import gradio as gr
from chatterbox.tts_turbo import ChatterboxTurboTTS

# --- setup & model loading ---

print("Checking for GPU...")
if not torch.cuda.is_available():
    print("⚠️ WARNING: No GPU detected. This model will be very slow on CPU.")
    print("Please go to Runtime > Change runtime type > T4 GPU.")
    device = "cpu"
else:
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    device = "cuda"

print("Downloading and Loading Chatterbox Turbo Model...")
# Initialize the model
MODEL = ChatterboxTurboTTS.from_pretrained(device)
print("✅ Model Loaded Successfully.")

# --- UI Constants & Helper Functions ---

EVENT_TAGS = [
    "[clear throat]", "[sigh]", "[shush]", "[cough]", "[groan]",
    "[sniff]", "[gasp]", "[chuckle]", "[laugh]"
]

CUSTOM_CSS = """
.tag-container {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 8px !important;
    margin-top: 5px !important;
    margin-bottom: 10px !important;
    border: none !important;
    background: transparent !important;
}

.tag-btn {
    min-width: fit-content !important;
    width: auto !important;
    height: 32px !important;
    font-size: 13px !important;
    background: #eef2ff !important;
    border: 1px solid #c7d2fe !important;
    color: #3730a3 !important;
    border-radius: 6px !important;
    padding: 0 10px !important;
    margin: 0 !important;
    box-shadow: none !important;
}

.tag-btn:hover {
    background: #c7d2fe !important;
    transform: translateY(-1px);
}
"""

INSERT_TAG_JS = """
(tag_val, current_text) => {
    const textarea = document.querySelector('#main_textbox textarea');
    if (!textarea) return current_text + " " + tag_val;

    const start = textarea.selectionStart;
    const end = textarea.selectionEnd;

    let prefix = " ";
    let suffix = " ";

    if (start === 0) prefix = "";
    else if (current_text[start - 1] === ' ') prefix = "";

    if (end < current_text.length && current_text[end] === ' ') suffix = "";

    return current_text.slice(0, start) + prefix + tag_val + suffix + current_text.slice(end);
}
"""

def set_seed(seed: int):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)

def generate(
        text,
        audio_prompt_path,
        temperature,
        seed_num,
        min_p,
        top_p,
        top_k,
        repetition_penalty,
        norm_loudness
):
    # If user provided a seed, set it
    if seed_num != 0:
        set_seed(int(seed_num))

    # Generate audio
    wav = MODEL.generate(
        text,
        audio_prompt_path=audio_prompt_path,
        temperature=temperature,
        min_p=min_p,
        top_p=top_p,
        top_k=int(top_k),
        repetition_penalty=repetition_penalty,
        norm_loudness=norm_loudness,
    )

    # Return sampling rate and audio array (cpu numpy)
    return (MODEL.sr, wav.squeeze(0).cpu().numpy())

# --- Gradio Interface ---

with gr.Blocks(title="Chatterbox Turbo", css=CUSTOM_CSS) as demo:
    gr.Markdown("# ⚡ Chatterbox Turbo (Colab Demo)")

    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                value="Oh, that's hilarious! [chuckle] Um anyway, we do have a new model in store. It's the SkyNet T-800 series and it's got basically everything. Including AI integration with ChatGPT and all that jazz. Would you like me to get some prices for you?",
                label="Text to synthesize (max chars 300)",
                max_lines=5,
                elem_id="main_textbox"
            )

            # Tag buttons
            with gr.Row(elem_classes=["tag-container"]):
                for tag in EVENT_TAGS:
                    btn = gr.Button(tag, elem_classes=["tag-btn"])
                    btn.click(
                        fn=None,
                        inputs=[btn, text],
                        outputs=text,
                        js=INSERT_TAG_JS
                    )

            ref_wav = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="Reference Audio File (Clone Voice)",
                value="https://storage.googleapis.com/chatterbox-demo-samples/prompts/Ethan.wav",
            )

            run_btn = gr.Button("Generate ⚡", variant="primary")

        with gr.Column():
            audio_output = gr.Audio(label="Output Audio")

            with gr.Accordion("Advanced Options", open=False):
                seed_num = gr.Number(value=0, label="Random seed (0 for random)")
                temp = gr.Slider(0.05, 2.0, step=.05, label="Temperature", value=0.8)
                top_p = gr.Slider(0.00, 1.00, step=0.01, label="Top P", value=0.95)
                top_k = gr.Slider(0, 1000, step=10, label="Top K", value=1000)
                repetition_penalty = gr.Slider(1.00, 2.00, step=0.05, label="Repetition Penalty", value=1.2)
                min_p = gr.Slider(0.00, 1.00, step=0.01, label="Min P (Set to 0 to disable)", value=0.00)
                norm_loudness = gr.Checkbox(value=True, label="Normalize Loudness (-27 LUFS)")

    run_btn.click(
        fn=generate,
        inputs=[
            text,
            ref_wav,
            temp,
            seed_num,
            min_p,
            top_p,
            top_k,
            repetition_penalty,
            norm_loudness,
        ],
        outputs=audio_output,
    )

# Launch with share=True for Colab
demo.queue().launch(share=True, ssr_mode=False)